# Pandas optimizations

## 1. Чтение fines.csv в датафрейм

In [38]:
import pandas as pd
import gc


input_file = '../data/fines.csv'

df = pd.read_csv(input_file, sep='\t')

df

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1,2100.0,Ford,Focus,1984
3,X582HE161RUS,2,2000.0,Ford,Focus,2015
4,92918M178RUS,1,5700.0,Ford,Focus,2014
...,...,...,...,...,...,...
925,Z000ZZ76RUS,1,1400.0,Ford,Focus,1981
926,B111BB76RUS,2,100.0,Lada,Granta,1999
927,L222LL76RUS,3,333.0,SAAB,900 turbo,2000
928,X333XX76RUS,4,219.0,Citroen,GT,2001


## 2. Итерации

- for i in range(0, len(df))

In [39]:
%%timeit

def get_calc_data_range():
    lst = []
    for i in range(0, len(df)):
        lst.append(df.Fines.iloc[i] / df.Refund.iloc[i] * df.Year.iloc[i])

    return lst

df["Culc_data"] = get_calc_data_range()

12 ms ± 612 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


- iterrows()

In [40]:
%%timeit

def get_calc_data_iterrows():
    lst = []
    for index, row in df.iterrows():
        lst.append(df.Fines[index] / df.Refund[index] * df.Year[index])

    return lst

df["Culc_data"] = get_calc_data_iterrows()

26 ms ± 746 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


- apply()

In [41]:
%%timeit

df["Culc_data"] = df.apply(lambda x: x["Fines"] / x["Refund"] * x["Year"], axis=1)

4.06 ms ± 164 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- Series объекты

In [42]:
%%timeit

df["Culc_data"] = df["Fines"] / df["Refund"] * df["Year"]

101 μs ± 2.4 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


- .values

In [43]:
%%timeit

df["Culc_data"] = df["Fines"].values / df["Refund"].values * df["Year"].values

44.5 μs ± 713 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 3. Индексирование

- Получение строки для определенного CarNumber

In [44]:
%%timeit

df[df["CarNumber"] == "O136HO197RUS"]

151 μs ± 10.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- Установка индекса с помощью CarNumber

In [45]:
df.set_index("CarNumber")

,Refund,Fines,Make,Model,Year,Culc_data
CarNumber,,,,,,
Y163O8161RUS,2,3200.0,Ford,Focus,1989,3182400.00
E432XX77RUS,1,6500.0,Toyota,Camry,1995,12967500.00
7184TT36RUS,1,2100.0,Ford,Focus,1984,4166400.00
X582HE161RUS,2,2000.0,Ford,Focus,2015,2015000.00
92918M178RUS,1,5700.0,Ford,Focus,2014,11479800.00
...,...,...,...,...,...,...
Z000ZZ76RUS,1,1400.0,Ford,Focus,1981,2773400.00
B111BB76RUS,2,100.0,Lada,Granta,1999,99950.00
L222LL76RUS,3,333.0,SAAB,900 turbo,2000,222000.00


- Снова получение строки CarNumber

In [46]:
%%timeit

df[df["CarNumber"] == "O136HO197RUS"]

145 μs ± 1.73 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 4. Даункастинг

- Запуск df.info(memory_usage='deep')

In [47]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    object 
 1   Refund     930 non-null    int64  
 2   Fines      930 non-null    float64
 3   Make       930 non-null    object 
 4   Model      917 non-null    object 
 5   Year       930 non-null    int64  
 6   Culc_data  930 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 182.1 KB


- Преобразование датафрейма в optimized_df

In [48]:
optimized_df = df.copy()

optimized_df

,CarNumber,Refund,Fines,Make,Model,Year,Culc_data
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989,3182400.00
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995,12967500.00
2,7184TT36RUS,1,2100.0,Ford,Focus,1984,4166400.00
3,X582HE161RUS,2,2000.0,Ford,Focus,2015,2015000.00
4,92918M178RUS,1,5700.0,Ford,Focus,2014,11479800.00
...,...,...,...,...,...,...,...
925,Z000ZZ76RUS,1,1400.0,Ford,Focus,1981,2773400.00
926,B111BB76RUS,2,100.0,Lada,Granta,1999,99950.00
927,L222LL76RUS,3,333.0,SAAB,900 turbo,2000,222000.00
928,X333XX76RUS,4,219.0,Citroen,GT,2001,109554.75


- Даункастинг от float64 к float32 для всех столбцов

In [49]:
optimized_df["Fines"] = optimized_df["Fines"].astype('float32')
optimized_df["Culc_data"] = optimized_df["Fines"].astype('float32')

optimized_df.dtypes

CarNumber     object
Refund         int64
Fines        float32
Make          object
Model         object
Year           int64
Culc_data    float32
dtype: object

- Понижение от int64 до наименьшего возможного числового значения Dtype

In [50]:
optimized_df["Refund"] = pd.to_numeric(optimized_df["Refund"], downcast='integer')
optimized_df["Year"] = pd.to_numeric(optimized_df["Year"], downcast='integer')

optimized_df.dtypes

CarNumber     object
Refund          int8
Fines        float32
Make          object
Model         object
Year           int16
Culc_data    float32
dtype: object

- Запуск info(memory_usage='deep')

In [51]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    object 
 1   Refund     930 non-null    int8   
 2   Fines      930 non-null    float32
 3   Make       930 non-null    object 
 4   Model      917 non-null    object 
 5   Year       930 non-null    int16  
 6   Culc_data  930 non-null    float32
dtypes: float32(2), int16(1), int8(1), object(3)
memory usage: 163.0 KB


## 5. Категории

- Изменение типа столбцов object на category

In [52]:
optimized_df["CarNumber"] = optimized_df["CarNumber"].astype('category')
optimized_df["Make"] = optimized_df["Make"].astype('category')
optimized_df["Model"] = optimized_df["Model"].astype('category')

- Проверка использования памяти

In [53]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  930 non-null    category
 1   Refund     930 non-null    int8    
 2   Fines      930 non-null    float32 
 3   Make       930 non-null    category
 4   Model      917 non-null    category
 5   Year       930 non-null    int16   
 6   Culc_data  930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 63.9 KB


## 6. Чистка памяти

In [54]:
gc.collect()

%reset_selective df